# Day 4.3 — Deterministic Checks Before Model Judgement
Some defects are facts: a parser can *prove* that a file calls `eval`, has a mutable default, or
swallows every exception. Proving costs nothing. Spend model calls on what is left.

### Step 1 — What a parser can prove

`ast` turns the source into a tree; walking it establishes three defects with no judgement. Note
the ids: a real analyser invents its own — which is why the evaluator matches by location.

In [ ]:
import ast

def deterministic_checks(source):
    """Walk the syntax tree and report the patterns a parser can establish as facts."""
    tree = ast.parse(source)                       # parse once; raises SyntaxError on bad input
    text_lines = source.splitlines()
    def excerpt(lineno):
        return text_lines[lineno - 1].strip() if 0 < lineno <= len(text_lines) else ""
    findings = []
    for node in ast.walk(tree):
        # 1) A direct call to eval() is a fact.
        if isinstance(node, ast.Call) and isinstance(node.func, ast.Name) and node.func.id == "eval":
            findings.append(Finding(id=f"AST-EVAL-{node.lineno}", category="security", line=node.lineno,
                                    title="Call to eval()", evidence=excerpt(node.lineno), severity="critical",
                                    recommendation="Use an allow-listed parser instead of eval.", reviewer="ast_checker"))
        # 2) A mutable default argument is visible in the signature.
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            if any(isinstance(d, (ast.List, ast.Dict, ast.Set)) for d in node.args.defaults):
                findings.append(Finding(id=f"AST-MUTABLE-DEFAULT-{node.lineno}", category="maintainability",
                                        line=node.lineno, title="Mutable default argument", evidence=excerpt(node.lineno),
                                        severity="medium", recommendation="Default to None and build the container inside.",
                                        reviewer="ast_checker"))
        # 3) `except Exception:` swallows unrelated failures.
        if isinstance(node, ast.ExceptHandler) and isinstance(node.type, ast.Name) and node.type.id == "Exception":
            findings.append(Finding(id=f"AST-BROAD-EXCEPT-{node.lineno}", category="maintainability",
                                    line=node.lineno, title="Broad exception handler", evidence=excerpt(node.lineno),
                                    severity="medium", recommendation="Catch only the exceptions you expect.",
                                    reviewer="ast_checker"))
    return sorted(findings, key=lambda f: (f.line, f.id))

checks = deterministic_checks(SOURCE)
print("Deterministic findings:", len(checks), "\n")
for finding in checks:
    print(f"line {finding.line:>3} | {finding.category:<15} | {finding.id:<26} | {finding.title}")
    print(f"         evidence: {finding.evidence}   -> golden defect {match_to_golden(finding)}")

### Step 2 — Measure what the proof cost

The whole argument for running tools first: the same evidence, for nothing, every time.

In [ ]:
start = perf_counter()
for _ in range(100):
    deterministic_checks(SOURCE)                   # 100 passes so the timer has something to see
print("Average time per AST pass  : %.3f ms" % ((perf_counter() - start) * 1000 / 100))
print("Model calls / tokens / cost: 0 / 0 / $0.000000")
print("Same answer on every run   : yes (same tree, same rules)")

### Step 3 — Checks plus one reviewer, with nothing merging them yet

Put the parser's three findings in front of the reviewer's five and score the pile. Recall rises
for zero extra tokens — and two defects are reported twice, because nothing merges them yet.

In [ ]:
reviewer_findings, reviewer_usage = call_reviewer(SOURCE, "general", reviewer_blind)
combined = checks + reviewer_findings              # plain concatenation: no merge rule at all
combined_row = score(combined)

print(f"{'system':<26}{'findings':>9}{'found':>7}{'dupes':>7}{'calls':>7}{'tokens':>8}")
print("-" * 64)
print(f"{'single_reviewer':<26}{len(reviewer_findings):>9}{baseline_row['found']:>7}"
      f"{baseline_row['duplicates']:>7}{1:>7}{single.total_tokens:>8}")
print(f"{'checks + reviewer (raw)':<26}{len(combined):>9}{combined_row['found']:>7}"
      f"{combined_row['duplicates']:>7}{1:>7}{single.total_tokens:>8}")
print("\nDefect gained by adding the parser:",
      sorted(set(baseline_row["missed"]) - set(combined_row["missed"])))
print("Extra model calls and tokens to gain it: 0 and 0")
print("Duplicates now in the report          :", combined_row["duplicates"], "- unmerged, and a reader has to triage them")

### Step 4 — Where the parser stops

None of the defects still missing are syntax facts. Each needs judgement about intent — where a
model earns its cost.

In [ ]:
print("Still missed after the parser ran:")
for defect_id in combined_row["missed"]:
    item = next(x for x in GOLDEN if x["id"] == defect_id)
    print(f"   {defect_id}  line {item['line']:>3}  {item['title']}")
print("\nBusiness rules, not syntax: a parser cannot know that a flat 20-unit discount is wrong.")

### Try it yourself

Of the parser's three findings, how many had the reviewer *already* reported? Predict, then run
the worked solution.

In [ ]:
# --- Worked solution ---------------------------------------------------------------
# Two findings are the same defect if they match the same golden id.
reviewer_defects = {match_to_golden(f) for f in reviewer_findings}
for finding in checks:
    defect = match_to_golden(finding)
    status = "already reported by the reviewer" if defect in reviewer_defects else "NEW"
    print(f"{finding.id:<26} -> {defect}  {status}")
print("\nOverlap is not waste here: it costs 0 tokens and upgrades a model's claim into a parser's proof.")
print("It becomes waste when you pay a second MODEL CALL to rediscover a fact ordinary code proved.")

### Checkpoint

**1. The parser and the reviewer both reported the `eval` call on line 20. Is that wasted work?**

<details><summary>Show answer</summary>

No: it is free (0 tokens) and upgrades a claim into a proof. It becomes waste when a *second model call* rediscovers what ordinary code already established.

</details>

**2. The deterministic step will write `model_calls: 0` into the trace instead of leaving the field out. Why?**

<details><summary>Show answer</summary>

An absent number gets filled in by whoever reads the table next. An explicit zero labelled "no model call" keeps the free step visible in every cost comparison.

</details>

### Recap

- **Limitation seen:** the reviewer missed defects a parser proves in under a millisecond.
- **Layer added:** an AST checker whose findings carry their own ids and are matched by location.
- **Evidence:** recall 5/9 → 6/9 for 0 extra calls — and 2 unmerged duplicates that now need a supervisor.